In [93]:
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from collections import defaultdict

# Before Feature Selection

In [2]:
file_path = "D:/hp/Documents/Practicum/Code/Dataset/kids_price.csv"

# Read the first line to count the number of columns
with open(file_path, 'r', encoding='utf-8') as x:
    ncols = len(x.readline().strip().split(','))

# Load CSV using the correct number of columns
df_kids = pd.read_csv(file_path, usecols=range(0, ncols))

# Display the DataFrame
df_kids.head()

,UMUR (BULAN),BANGSA,AGAMA,JANTINA,PENDAPATAN KELUARGA,GAJI BAPA,GAJI IBU,GAJI PENJAGA,STATUS PEMAKANAN,DAERAH,...,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,Melayu,Islam,LELAKI,M40,"RM 4,000 - RM 6,999","RM 4,000 - RM 6,999","RM 4,000 - RM 6,999",Malpemakanan,Seberang Perai Tengah,...,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,54,Melayu,Islam,LELAKI,Tiada Maklumat,"RM 10,000 dan ke atas","RM 4,000 - RM 6,999","RM 4,000 - RM 6,999",Malpemakanan,Seberang Perai Tengah,...,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,53,Melayu,Islam,PEREMPUAN,B40,"RM 1,000 - RM 3,999","RM 1,000 - RM 3,999","RM 1,000 - RM 3,999",Malpemakanan,Seberang Perai Tengah,...,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,52,Melayu,Islam,PEREMPUAN,B40,"RM 1,000 - RM 3,999","RM 1,000 - RM 3,999","RM 1,000 - RM 3,999",Malpemakanan,Seberang Perai Tengah,...,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
4,58,Melayu,Islam,LELAKI,B40,"RM 1,000 - RM 3,999",TIADA MAKLUMAT GAJI,"RM 1,000 - RM 3,999",Malpemakanan,Daerah Barat Daya,...,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


## Encoding Strategy

### Label Encoder

In [3]:
# Create a LabelEncoder object
le = LabelEncoder()
df_kids_le = df_kids.copy()

# Iterate over the columns of the DataFrame
for col in df_kids_le.columns:
    # Check if the column is of object type (categorical)
    if df_kids_le[col].dtype == 'object':
        # Fit and transform the column using LabelEncoder
        df_kids_le[col] = le.fit_transform(df_kids_le[col])

df_kids_le.head()

,UMUR (BULAN),BANGSA,AGAMA,JANTINA,PENDAPATAN KELUARGA,GAJI BAPA,GAJI IBU,GAJI PENJAGA,STATUS PEMAKANAN,DAERAH,...,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,5,2,0,1,4,4,4,1,3,...,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,54,5,2,0,4,3,4,4,1,3,...,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,53,5,2,1,0,2,2,2,1,3,...,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,52,5,2,1,0,2,2,2,1,3,...,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
4,58,5,2,0,0,2,6,2,1,0,...,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


In [4]:
x_le = df_kids_le.drop('BMI', axis=1)
y_le = df_kids_le['BMI']

x_train_le, x_test_le, y_train_le, y_test_le = train_test_split( x_le, y_le, test_size = 0.30, random_state=42)

In [5]:
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(x_train_le, y_train_le)

# Make predictions on the test set
predictions = rf_model.predict(x_test_le)

In [6]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_le, predictions))

print("\nClassification Report:")
print(classification_report(y_test_le, predictions))

Confusion Matrix:
[[34  0  6  9  6  1]
 [10  5  9 12  6  1]
 [ 3  3  6  4  5  3]
 [14  6  7 18  4  2]
 [ 3  5  1  3  5  1]
 [ 3  1  2  0  6  2]]

Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.61      0.55        56
           1       0.25      0.12      0.16        43
           2       0.19      0.25      0.22        24
           3       0.39      0.35      0.37        51
           4       0.16      0.28      0.20        18
           5       0.20      0.14      0.17        14

    accuracy                           0.34       206
   macro avg       0.28      0.29      0.28       206
weighted avg       0.34      0.34      0.33       206



### One-Hot Encoder

In [7]:
df_kids_ohe = df_kids.copy()
df_kids_ohe = df_kids_ohe.drop('BMI', axis=1)

In [8]:
# Create a OneHotEncoder object
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform the categorical columns
categorical_cols = df_kids_ohe.select_dtypes(include=['object']).columns
df_kids_encoded = pd.DataFrame(ohe.fit_transform(df_kids_ohe[categorical_cols]))

# Get feature names for the encoded columns
encoded_feature_names = list(ohe.get_feature_names_out(categorical_cols))
df_kids_encoded.columns = encoded_feature_names

# Drop original categorical columns from the dataframe
df_kids_ohe = df_kids_ohe.drop(categorical_cols, axis=1)

# Concatenate the encoded columns with the remaining numerical features
df_kids_ohe = pd.concat([df_kids_ohe, df_kids_encoded], axis=1)

df_kids_ohe.head()

,UMUR (BULAN),banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,...,DAERAH_Seberang Perai Selatan,DAERAH_Seberang Perai Tengah,DAERAH_Seberang Perai Utara,"JENIS TASKA_TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, YPKT)",JENIS TASKA_TASKA Di Rumah,JENIS TASKA_TASKA Di Tempat Kerja (Sektor Awam),JENIS TASKA_TASKA Di Tempat Kerja (Sektor Swasta),JENIS TASKA_TASKA Institusi,TASKA_LOKASI_BANDAR,TASKA_LOKASI_LUAR BANDAR
0,60,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,54,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
2,53,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,52,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4,58,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [9]:
x_ohe = df_kids_ohe
y_ohe = df_kids['BMI']

x_train_ohe, x_test_ohe, y_train_ohe, y_test_ohe = train_test_split( x_ohe, y_ohe, test_size = 0.30, random_state=42)

In [10]:
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(x_train_ohe, y_train_ohe)

# Make predictions on the test set
predictions = rf_model.predict(x_test_ohe)

In [11]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_ohe, predictions))

print("\nClassification Report:")
print(classification_report(y_test_ohe, predictions))

Confusion Matrix:
[[32  0  8  9  6  1]
 [ 9  4 10 13  6  1]
 [ 3  3  6  4  6  2]
 [14  6  7 18  4  2]
 [ 3  3  1  5  5  1]
 [ 4  0  2  0  6  2]]

Classification Report:
                               precision    recall  f1-score   support

           Berat badan normal       0.49      0.57      0.53        56
       Berlebihan berat badan       0.25      0.09      0.14        43
                         Obes       0.18      0.25      0.21        24
Risiko berlebihan berat badan       0.37      0.35      0.36        51
                        Susut       0.15      0.28      0.20        18
                  Susut teruk       0.22      0.14      0.17        14

                     accuracy                           0.33       206
                    macro avg       0.28      0.28      0.27       206
                 weighted avg       0.33      0.33      0.31       206



## Use SMOTE(Synthetic Minority Over-sampling Technique) to solve class imbalance

### Label Encoder

In [12]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(x_train_le, y_train_le)

In [13]:
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_res, y_train_res)

# Make predictions on the test set
predictions = rf_model.predict(x_test_le)

In [14]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_le, predictions))

print("\nClassification Report:")
print(classification_report(y_test_le, predictions))

Confusion Matrix:
[[30  1  6 12  5  2]
 [ 9  3  7 13  7  4]
 [ 3  4  6  3  4  4]
 [11 10  5 17  5  3]
 [ 1  4  1  3  6  3]
 [ 3  0  3  0  6  2]]

Classification Report:
              precision    recall  f1-score   support

           0       0.53      0.54      0.53        56
           1       0.14      0.07      0.09        43
           2       0.21      0.25      0.23        24
           3       0.35      0.33      0.34        51
           4       0.18      0.33      0.24        18
           5       0.11      0.14      0.12        14

    accuracy                           0.31       206
   macro avg       0.25      0.28      0.26       206
weighted avg       0.31      0.31      0.30       206



### One-Hot Encoder

In [15]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(x_train_ohe, y_train_ohe)

In [16]:
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_res, y_train_res)

# Make predictions on the test set
predictions = rf_model.predict(x_test_ohe)

In [17]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_ohe, predictions))

print("\nClassification Report:")
print(classification_report(y_test_ohe, predictions))

Confusion Matrix:
[[32  0  7 10  6  1]
 [ 8  4  9 15  6  1]
 [ 3  3  6  4  4  4]
 [14  5  8 18  3  3]
 [ 2  2  3  4  5  2]
 [ 3  1  3  0  5  2]]

Classification Report:
                               precision    recall  f1-score   support

           Berat badan normal       0.52      0.57      0.54        56
       Berlebihan berat badan       0.27      0.09      0.14        43
                         Obes       0.17      0.25      0.20        24
Risiko berlebihan berat badan       0.35      0.35      0.35        51
                        Susut       0.17      0.28      0.21        18
                  Susut teruk       0.15      0.14      0.15        14

                     accuracy                           0.33       206
                    macro avg       0.27      0.28      0.27       206
                 weighted avg       0.33      0.33      0.32       206



# After Feature Selection

## Strategy: Drop Invalid Data

In [18]:
file_path = "D:/hp/Documents/Practicum/Code/Dataset/DROP_INVALID_AfterFS.csv"

# Read the first line to count the number of columns
with open(file_path, 'r', encoding='utf-8') as x:
    ncols = len(x.readline().strip().split(','))

# Load CSV using the correct number of columns
df_DROP_INVALID_AfterFS = pd.read_csv(file_path, usecols=range(0, ncols))

# Display the DataFrame
df_DROP_INVALID_AfterFS.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,Malpemakanan,TASKA Institusi,BANDAR,Risiko berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,53,Malpemakanan,TASKA Di Rumah,BANDAR,Risiko berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,52,Malpemakanan,TASKA Institusi,BANDAR,Obes,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,58,Malpemakanan,"TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, Y...",LUAR BANDAR,Berlebihan berat badan,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9
4,56,Malpemakanan,"TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, Y...",LUAR BANDAR,Berat badan normal,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


### Encoding Strategy

#### Label Encoder

In [19]:
# Create a LabelEncoder object
le = LabelEncoder()
df_DROP_INVALID_AfterFS_le = df_DROP_INVALID_AfterFS.copy()

# Iterate over the columns of the DataFrame
for col in df_DROP_INVALID_AfterFS_le.columns:
    # Check if the column is of object type (categorical)
    if df_DROP_INVALID_AfterFS_le[col].dtype == 'object':
        # Fit and transform the column using LabelEncoder
        df_DROP_INVALID_AfterFS_le[col] = le.fit_transform(df_DROP_INVALID_AfterFS_le[col])

df_DROP_INVALID_AfterFS_le.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,0,4,0,3,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,53,0,1,0,3,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,52,0,4,0,2,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,58,0,0,1,1,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9
4,56,0,0,1,0,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


In [20]:
x_le = df_DROP_INVALID_AfterFS_le.drop('BMI', axis=1)
y_le = df_DROP_INVALID_AfterFS_le['BMI']

x_train_le, x_test_le, y_train_le, y_test_le = train_test_split( x_le, y_le, test_size = 0.30, random_state=42)

In [21]:
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(x_train_le, y_train_le)

# Make predictions on the test set
predictions = rf_model.predict(x_test_le)

In [22]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_le, predictions))

print("\nClassification Report:")
print(classification_report(y_test_le, predictions))

Confusion Matrix:
[[33  2  8  9  4  0]
 [ 8  4  3 11  1  1]
 [ 6  2  8  3  1  2]
 [ 4  7  6 13  3  1]
 [ 4  1  3  3  6  0]
 [ 3  1  2  2  1  1]]

Classification Report:
              precision    recall  f1-score   support

           0       0.57      0.59      0.58        56
           1       0.24      0.14      0.18        28
           2       0.27      0.36      0.31        22
           3       0.32      0.38      0.35        34
           4       0.38      0.35      0.36        17
           5       0.20      0.10      0.13        10

    accuracy                           0.39       167
   macro avg       0.33      0.32      0.32       167
weighted avg       0.38      0.39      0.38       167



#### One-Hot Encoder

In [23]:
df_DROP_INVALID_AfterFS_ohe = df_DROP_INVALID_AfterFS.copy()
df_DROP_INVALID_AfterFS_ohe = df_DROP_INVALID_AfterFS_ohe.drop('BMI', axis=1)

In [24]:
# Create a OneHotEncoder object
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform the categorical columns
categorical_cols = df_DROP_INVALID_AfterFS_ohe.select_dtypes(include=['object']).columns
df_DROP_INVALID_AfterFS_encoded = pd.DataFrame(ohe.fit_transform(df_DROP_INVALID_AfterFS_ohe[categorical_cols]))

# Get feature names for the encoded columns
encoded_feature_names = list(ohe.get_feature_names_out(categorical_cols))
df_DROP_INVALID_AfterFS_encoded.columns = encoded_feature_names

# Drop original categorical columns from the dataframe
df_DROP_INVALID_AfterFS_ohe = df_DROP_INVALID_AfterFS_ohe.drop(categorical_cols, axis=1)

# Concatenate the encoded columns with the remaining numerical features
df_DROP_INVALID_AfterFS_ohe = pd.concat([df_DROP_INVALID_AfterFS_ohe, df_DROP_INVALID_AfterFS_encoded], axis=1)

df_DROP_INVALID_AfterFS_ohe.head()

,UMUR (BULAN),banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price,STATUS PEMAKANAN_Malpemakanan,STATUS PEMAKANAN_Normal,"JENIS TASKA_TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, YPKT)",JENIS TASKA_TASKA Di Rumah,JENIS TASKA_TASKA Di Tempat Kerja (Sektor Awam),JENIS TASKA_TASKA Di Tempat Kerja (Sektor Swasta),JENIS TASKA_TASKA Institusi,TASKA_LOKASI_BANDAR,TASKA_LOKASI_LUAR BANDAR
0,60,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,53,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,52,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,58,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4,56,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [25]:
x_ohe = df_DROP_INVALID_AfterFS_ohe
y_ohe = df_DROP_INVALID_AfterFS['BMI']

x_train_ohe, x_test_ohe, y_train_ohe, y_test_ohe = train_test_split( x_ohe, y_ohe, test_size = 0.30, random_state=42)

In [26]:
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(x_train_ohe, y_train_ohe)

# Make predictions on the test set
predictions = rf_model.predict(x_test_ohe)

In [27]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_ohe, predictions))

print("\nClassification Report:")
print(classification_report(y_test_ohe, predictions))

Confusion Matrix:
[[34  3  7  9  3  0]
 [ 8  4  3 11  1  1]
 [ 6  2  8  3  1  2]
 [ 4  7  6 13  3  1]
 [ 3  1  3  4  6  0]
 [ 3  1  3  1  1  1]]

Classification Report:
                               precision    recall  f1-score   support

           Berat badan normal       0.59      0.61      0.60        56
       Berlebihan berat badan       0.22      0.14      0.17        28
                         Obes       0.27      0.36      0.31        22
Risiko berlebihan berat badan       0.32      0.38      0.35        34
                        Susut       0.40      0.35      0.38        17
                  Susut teruk       0.20      0.10      0.13        10

                     accuracy                           0.40       167
                    macro avg       0.33      0.32      0.32       167
                 weighted avg       0.39      0.40      0.39       167



### Use SMOTE(Synthetic Minority Over-sampling Technique) to solve class imbalance

#### Label Encoder

In [28]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(x_train_le, y_train_le)

In [29]:
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_res, y_train_res)

# Make predictions on the test set
predictions = rf_model.predict(x_test_le)

In [30]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_le, predictions))

print("\nClassification Report:")
print(classification_report(y_test_le, predictions))

Confusion Matrix:
[[32  3  7 10  4  0]
 [ 7  4  3 11  3  0]
 [ 5  3  8  2  2  2]
 [ 4  8  6 13  2  1]
 [ 4  1  4  2  6  0]
 [ 2  1  3  1  1  2]]

Classification Report:
              precision    recall  f1-score   support

           0       0.59      0.57      0.58        56
           1       0.20      0.14      0.17        28
           2       0.26      0.36      0.30        22
           3       0.33      0.38      0.36        34
           4       0.33      0.35      0.34        17
           5       0.40      0.20      0.27        10

    accuracy                           0.39       167
   macro avg       0.35      0.34      0.34       167
weighted avg       0.39      0.39      0.39       167



#### One-Hot Encoder

In [31]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(x_train_ohe, y_train_ohe)

In [32]:
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_res, y_train_res)

# Make predictions on the test set
predictions = rf_model.predict(x_test_ohe)

In [33]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_ohe, predictions))

print("\nClassification Report:")
print(classification_report(y_test_ohe, predictions))

Confusion Matrix:
[[32  4  7 11  2  0]
 [ 8  4  3 10  2  1]
 [ 5  4  8  2  2  1]
 [ 3  9  6 12  3  1]
 [ 4  1  3  3  6  0]
 [ 2  1  3  1  1  2]]

Classification Report:
                               precision    recall  f1-score   support

           Berat badan normal       0.59      0.57      0.58        56
       Berlebihan berat badan       0.17      0.14      0.16        28
                         Obes       0.27      0.36      0.31        22
Risiko berlebihan berat badan       0.31      0.35      0.33        34
                        Susut       0.38      0.35      0.36        17
                  Susut teruk       0.40      0.20      0.27        10

                     accuracy                           0.38       167
                    macro avg       0.35      0.33      0.33       167
                 weighted avg       0.39      0.38      0.38       167



## Strategy: Replace Invalid Data with Mode

In [34]:
file_path = "D:/hp/Documents/Practicum/Code/Dataset/RP_MODE_AfterFS.csv"

# Read the first line to count the number of columns
with open(file_path, 'r', encoding='utf-8') as x:
    ncols = len(x.readline().strip().split(','))

# Load CSV using the correct number of columns
df_RP_MODE_AfterFS = pd.read_csv(file_path, usecols=range(0, ncols))

# Display the DataFrame
df_RP_MODE_AfterFS.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,Malpemakanan,TASKA Institusi,BANDAR,Risiko berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,54,Malpemakanan,TASKA Institusi,BANDAR,Berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,53,Malpemakanan,TASKA Di Rumah,BANDAR,Risiko berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,52,Malpemakanan,TASKA Institusi,BANDAR,Obes,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
4,58,Malpemakanan,"TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, Y...",LUAR BANDAR,Berlebihan berat badan,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


### Encoding Strategy

#### Label Encoder

In [35]:
# Create a LabelEncoder object
le = LabelEncoder()
df_RP_MODE_AfterFS_le = df_RP_MODE_AfterFS.copy()

# Iterate over the columns of the DataFrame
for col in df_RP_MODE_AfterFS_le.columns:
    # Check if the column is of object type (categorical)
    if df_RP_MODE_AfterFS_le[col].dtype == 'object':
        # Fit and transform the column using LabelEncoder
        df_RP_MODE_AfterFS_le[col] = le.fit_transform(df_RP_MODE_AfterFS_le[col])

df_RP_MODE_AfterFS_le.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,1,4,0,3,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,54,1,4,0,1,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,53,1,1,0,3,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,52,1,4,0,2,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
4,58,1,0,1,1,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


In [36]:
x_le = df_RP_MODE_AfterFS_le.drop('BMI', axis=1)
y_le = df_RP_MODE_AfterFS_le['BMI']

x_train_le, x_test_le, y_train_le, y_test_le = train_test_split( x_le, y_le, test_size = 0.30, random_state=42)

In [37]:
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(x_train_le, y_train_le)

# Make predictions on the test set
predictions = rf_model.predict(x_test_le)

In [38]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_le, predictions))

print("\nClassification Report:")
print(classification_report(y_test_le, predictions))

Confusion Matrix:
[[26  4  3 10  9  4]
 [ 8  6  8 13  3  5]
 [ 6  2  6  5  4  1]
 [12  8  6 18  5  2]
 [ 2  1  2  6  7  0]
 [ 2  1  1  4  6  0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.46      0.46      0.46        56
           1       0.27      0.14      0.18        43
           2       0.23      0.25      0.24        24
           3       0.32      0.35      0.34        51
           4       0.21      0.39      0.27        18
           5       0.00      0.00      0.00        14

    accuracy                           0.31       206
   macro avg       0.25      0.27      0.25       206
weighted avg       0.31      0.31      0.30       206



#### One-Hot Encoder

In [39]:
df_RP_MODE_AfterFS_ohe = df_RP_MODE_AfterFS.copy()
df_RP_MODE_AfterFS_ohe = df_RP_MODE_AfterFS_ohe.drop('BMI', axis=1)

In [40]:
# Create a OneHotEncoder object
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform the categorical columns
categorical_cols = df_RP_MODE_AfterFS_ohe.select_dtypes(include=['object']).columns
df_RP_MODE_AfterFS_encoded = pd.DataFrame(ohe.fit_transform(df_RP_MODE_AfterFS_ohe[categorical_cols]))

# Get feature names for the encoded columns
encoded_feature_names = list(ohe.get_feature_names_out(categorical_cols))
df_RP_MODE_AfterFS_encoded.columns = encoded_feature_names

# Drop original categorical columns from the dataframe
df_RP_MODE_AfterFS_ohe = df_RP_MODE_AfterFS_ohe.drop(categorical_cols, axis=1)

# Concatenate the encoded columns with the remaining numerical features
df_RP_MODE_AfterFS_ohe = pd.concat([df_RP_MODE_AfterFS_ohe, df_RP_MODE_AfterFS_encoded], axis=1)

df_RP_MODE_AfterFS_ohe.head()

,UMUR (BULAN),banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,...,STATUS PEMAKANAN_Error/Tiada Data,STATUS PEMAKANAN_Malpemakanan,STATUS PEMAKANAN_Normal,"JENIS TASKA_TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, YPKT)",JENIS TASKA_TASKA Di Rumah,JENIS TASKA_TASKA Di Tempat Kerja (Sektor Awam),JENIS TASKA_TASKA Di Tempat Kerja (Sektor Swasta),JENIS TASKA_TASKA Institusi,TASKA_LOKASI_BANDAR,TASKA_LOKASI_LUAR BANDAR
0,60,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,54,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
2,53,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,52,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4,58,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [41]:
x_ohe = df_RP_MODE_AfterFS_ohe
y_ohe = df_RP_MODE_AfterFS['BMI']

x_train_ohe, x_test_ohe, y_train_ohe, y_test_ohe = train_test_split( x_ohe, y_ohe, test_size = 0.30, random_state=42)

In [42]:
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(x_train_ohe, y_train_ohe)

# Make predictions on the test set
predictions = rf_model.predict(x_test_ohe)

In [43]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_ohe, predictions))

print("\nClassification Report:")
print(classification_report(y_test_ohe, predictions))

Confusion Matrix:
[[26  4  4  9  9  4]
 [ 7  7  9 13  2  5]
 [ 5  2  6  5  5  1]
 [12  9  5 19  4  2]
 [ 3  1  2  5  7  0]
 [ 2  1  1  4  6  0]]

Classification Report:
                               precision    recall  f1-score   support

           Berat badan normal       0.47      0.46      0.47        56
       Berlebihan berat badan       0.29      0.16      0.21        43
                         Obes       0.22      0.25      0.24        24
Risiko berlebihan berat badan       0.35      0.37      0.36        51
                        Susut       0.21      0.39      0.27        18
                  Susut teruk       0.00      0.00      0.00        14

                     accuracy                           0.32       206
                    macro avg       0.26      0.27      0.26       206
                 weighted avg       0.32      0.32      0.31       206



### Use SMOTE(Synthetic Minority Over-sampling Technique) to solve class imbalance

#### Label Encoder

In [44]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(x_train_le, y_train_le)

In [45]:
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_res, y_train_res)

# Make predictions on the test set
predictions = rf_model.predict(x_test_le)

In [46]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_le, predictions))

print("\nClassification Report:")
print(classification_report(y_test_le, predictions))

Confusion Matrix:
[[25  5  4  9  9  4]
 [ 8  9  6 13  1  6]
 [ 7  2  5  4  5  1]
 [11 10  4 20  4  2]
 [ 2  3  1  5  7  0]
 [ 2  1  2  4  5  0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.45      0.45      0.45        56
           1       0.30      0.21      0.25        43
           2       0.23      0.21      0.22        24
           3       0.36      0.39      0.38        51
           4       0.23      0.39      0.29        18
           5       0.00      0.00      0.00        14

    accuracy                           0.32       206
   macro avg       0.26      0.27      0.26       206
weighted avg       0.32      0.32      0.32       206



#### One-Hot Encoder

In [47]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(x_train_ohe, y_train_ohe)

In [48]:
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_res, y_train_res)

# Make predictions on the test set
predictions = rf_model.predict(x_test_ohe)

In [49]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_ohe, predictions))

print("\nClassification Report:")
print(classification_report(y_test_ohe, predictions))

Confusion Matrix:
[[26  4  3  9 10  4]
 [ 8  9  7 13  1  5]
 [ 8  2  6  3  4  1]
 [11 11  5 19  3  2]
 [ 3  1  2  3  7  2]
 [ 2  2  1  4  5  0]]

Classification Report:
                               precision    recall  f1-score   support

           Berat badan normal       0.45      0.46      0.46        56
       Berlebihan berat badan       0.31      0.21      0.25        43
                         Obes       0.25      0.25      0.25        24
Risiko berlebihan berat badan       0.37      0.37      0.37        51
                        Susut       0.23      0.39      0.29        18
                  Susut teruk       0.00      0.00      0.00        14

                     accuracy                           0.33       206
                    macro avg       0.27      0.28      0.27       206
                 weighted avg       0.33      0.33      0.32       206



# Model Tuning

In [35]:
file_path = "D:/hp/Documents/Practicum/Code/Dataset/DROP_INVALID_AfterFS.csv"

# Read the first line to count the number of columns
with open(file_path, 'r', encoding='utf-8') as x:
    ncols = len(x.readline().strip().split(','))

# Load CSV using the correct number of columns
df_DROP_INVALID_AfterFS = pd.read_csv(file_path, usecols=range(0, ncols))

# Display the DataFrame
df_DROP_INVALID_AfterFS.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,Malpemakanan,TASKA Institusi,BANDAR,Risiko berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,53,Malpemakanan,TASKA Di Rumah,BANDAR,Risiko berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,52,Malpemakanan,TASKA Institusi,BANDAR,Obes,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,58,Malpemakanan,"TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, Y...",LUAR BANDAR,Berlebihan berat badan,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9
4,56,Malpemakanan,"TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, Y...",LUAR BANDAR,Berat badan normal,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


In [36]:
df_DROP_INVALID_AfterFS_ohe = df_DROP_INVALID_AfterFS.copy()
df_DROP_INVALID_AfterFS_ohe = df_DROP_INVALID_AfterFS_ohe.drop('BMI', axis=1)

In [37]:
# Create a OneHotEncoder object
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform the categorical columns
categorical_cols = df_DROP_INVALID_AfterFS_ohe.select_dtypes(include=['object']).columns
df_DROP_INVALID_AfterFS_encoded = pd.DataFrame(ohe.fit_transform(df_DROP_INVALID_AfterFS_ohe[categorical_cols]))

# Get feature names for the encoded columns
encoded_feature_names = list(ohe.get_feature_names_out(categorical_cols))
df_DROP_INVALID_AfterFS_encoded.columns = encoded_feature_names

# Drop original categorical columns from the dataframe
df_DROP_INVALID_AfterFS_ohe = df_DROP_INVALID_AfterFS_ohe.drop(categorical_cols, axis=1)

# Concatenate the encoded columns with the remaining numerical features
df_DROP_INVALID_AfterFS_ohe = pd.concat([df_DROP_INVALID_AfterFS_ohe, df_DROP_INVALID_AfterFS_encoded], axis=1)

df_DROP_INVALID_AfterFS_ohe.head()

,UMUR (BULAN),banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price,STATUS PEMAKANAN_Malpemakanan,STATUS PEMAKANAN_Normal,"JENIS TASKA_TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, YPKT)",JENIS TASKA_TASKA Di Rumah,JENIS TASKA_TASKA Di Tempat Kerja (Sektor Awam),JENIS TASKA_TASKA Di Tempat Kerja (Sektor Swasta),JENIS TASKA_TASKA Institusi,TASKA_LOKASI_BANDAR,TASKA_LOKASI_LUAR BANDAR
0,60,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,53,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,52,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,58,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4,56,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [53]:
x_ohe = df_DROP_INVALID_AfterFS_ohe
y_ohe = df_DROP_INVALID_AfterFS['BMI']

x_train, x_test, y_train, y_test = train_test_split( x_ohe, y_ohe, test_size = 0.30, random_state=42)

In [54]:
# Define the parameter grid to search
param_grid = {
    'n_estimators': [50, 100, 200, 300, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'bootstrap': [True, False],
    'class_weight': [None, 'balanced']
}

In [ ]:
grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42),
                           param_grid=param_grid,
                           cv=5,
                           scoring='accuracy',
                           n_jobs=-1,
                           verbose=2)

grid_search.fit(x_train, y_train)

Fitting 5 folds for each of 2160 candidates, totalling 10800 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'bootstrap': [True, False],
                         'class_weight': [None, 'balanced'],
                         'max_depth': [None, 10, 20, 30],
                         'max_features': [None, 'sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 200, 300, 500]},
             scoring='accuracy', verbose=2)

In [ ]:
# Print the best parameters and best score
print("Best parameters found: ", grid_search.best_params_)
print("Best accuracy found: ", grid_search.best_score_)

Best parameters found:  {'bootstrap': False, 'class_weight': None, 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 100}
Best accuracy found:  0.4601731601731601


In [ ]:
# Get the best model from the grid search
best_rf_model = grid_search.best_estimator_

# Make predictions on the test set using the best model
tuned_predictions = best_rf_model.predict(x_test)

# Evaluate the tuned model
print("\nTuned Model Confusion Matrix:")
print(confusion_matrix(y_test, tuned_predictions))

print("\nTuned Model Classification Report:")
print(classification_report(y_test, tuned_predictions))


Tuned Model Confusion Matrix:
[[37  3  5 10  1  0]
 [ 4  3  3 17  1  0]
 [ 4  2  5  9  1  1]
 [ 8  3  4 18  0  1]
 [ 6  0  3  1  6  1]
 [ 3  1  1  3  1  1]]

Tuned Model Classification Report:
                               precision    recall  f1-score   support

           Berat badan normal       0.60      0.66      0.63        56
       Berlebihan berat badan       0.25      0.11      0.15        28
                         Obes       0.24      0.23      0.23        22
Risiko berlebihan berat badan       0.31      0.53      0.39        34
                        Susut       0.60      0.35      0.44        17
                  Susut teruk       0.25      0.10      0.14        10

                     accuracy                           0.42       167
                    macro avg       0.37      0.33      0.33       167
                 weighted avg       0.41      0.42      0.40       167



# Number of Target Labels

In [38]:
file_path = "D:/hp/Documents/Practicum/Code/Dataset/DROP_INVALID_AfterFS.csv"

# Read the first line to count the number of columns
with open(file_path, 'r', encoding='utf-8') as x:
    ncols = len(x.readline().strip().split(','))

# Load CSV using the correct number of columns
df_DROP_INVALID_AfterFS = pd.read_csv(file_path, usecols=range(0, ncols))

# Display the DataFrame
df_DROP_INVALID_AfterFS.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,Malpemakanan,TASKA Institusi,BANDAR,Risiko berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,53,Malpemakanan,TASKA Di Rumah,BANDAR,Risiko berlebihan berat badan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,52,Malpemakanan,TASKA Institusi,BANDAR,Obes,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,58,Malpemakanan,"TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, Y...",LUAR BANDAR,Berlebihan berat badan,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9
4,56,Malpemakanan,"TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, Y...",LUAR BANDAR,Berat badan normal,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


## 4 Labels

In [56]:
df = df_DROP_INVALID_AfterFS.replace({'BMI': {"Berat badan normal": "Normal",
                                                 "Berlebihan berat badan": "Berat berlebihan",
                                                 "Obes": "Berat berlebihan",
                                                 "Risiko berlebihan berat badan": "Berisiko",
                                                 "Susut": "Kurang berat badan",
                                                 "Susut teruk": "Kurang berat badan"}})

In [57]:
df_DROP_INVALID_AfterFS_ohe = df.copy()
df_DROP_INVALID_AfterFS_ohe = df_DROP_INVALID_AfterFS_ohe.drop('BMI', axis=1)

In [58]:
# Create a OneHotEncoder object
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform the categorical columns
categorical_cols = df_DROP_INVALID_AfterFS_ohe.select_dtypes(include=['object']).columns
df_DROP_INVALID_AfterFS_encoded = pd.DataFrame(ohe.fit_transform(df_DROP_INVALID_AfterFS_ohe[categorical_cols]))

# Get feature names for the encoded columns
encoded_feature_names = list(ohe.get_feature_names_out(categorical_cols))
df_DROP_INVALID_AfterFS_encoded.columns = encoded_feature_names

# Drop original categorical columns from the dataframe
df_DROP_INVALID_AfterFS_ohe = df_DROP_INVALID_AfterFS_ohe.drop(categorical_cols, axis=1)

# Concatenate the encoded columns with the remaining numerical features
df_DROP_INVALID_AfterFS_ohe = pd.concat([df_DROP_INVALID_AfterFS_ohe, df_DROP_INVALID_AfterFS_encoded], axis=1)

df_DROP_INVALID_AfterFS_ohe.head()

,UMUR (BULAN),banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price,STATUS PEMAKANAN_Malpemakanan,STATUS PEMAKANAN_Normal,"JENIS TASKA_TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, YPKT)",JENIS TASKA_TASKA Di Rumah,JENIS TASKA_TASKA Di Tempat Kerja (Sektor Awam),JENIS TASKA_TASKA Di Tempat Kerja (Sektor Swasta),JENIS TASKA_TASKA Institusi,TASKA_LOKASI_BANDAR,TASKA_LOKASI_LUAR BANDAR
0,60,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,53,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,52,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,58,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4,56,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [59]:
x_ohe = df_DROP_INVALID_AfterFS_ohe
y_ohe = df['BMI']

x_train_ohe, x_test_ohe, y_train_ohe, y_test_ohe = train_test_split(x_ohe, y_ohe, test_size = 0.30, random_state=42)

In [60]:
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(bootstrap=False, class_weight=None, max_depth=10, max_features='sqrt', min_samples_leaf=4,
                                  min_samples_split=10, n_estimators=100)
rf_model.fit(x_train_ohe, y_train_ohe)

# Make predictions on the test set
predictions = rf_model.predict(x_test_ohe)

In [61]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_ohe, predictions))

print("\nClassification Report:")
print(classification_report(y_test_ohe, predictions))

Confusion Matrix:
[[20 22  3  5]
 [ 8 17  2  7]
 [ 7  4  9  7]
 [12  8  1 35]]

Classification Report:
                    precision    recall  f1-score   support

  Berat berlebihan       0.43      0.40      0.41        50
          Berisiko       0.33      0.50      0.40        34
Kurang berat badan       0.60      0.33      0.43        27
            Normal       0.65      0.62      0.64        56

          accuracy                           0.49       167
         macro avg       0.50      0.46      0.47       167
      weighted avg       0.51      0.49      0.49       167



## 3 Labels

In [78]:
df = df_DROP_INVALID_AfterFS.replace({'BMI': {"Berat badan normal": "Normal",
                                                 "Berlebihan berat badan": "Berat berlebihan",
                                                 "Obes": "Berat berlebihan",
                                                 "Risiko berlebihan berat badan": "Berat berlebihan",
                                                 "Susut": "Kurang berat badan",
                                                 "Susut teruk": "Kurang berat badan"}})

In [79]:
df_DROP_INVALID_AfterFS_ohe = df.copy()
df_DROP_INVALID_AfterFS_ohe = df_DROP_INVALID_AfterFS_ohe.drop('BMI', axis=1)

In [80]:
# Create a OneHotEncoder object
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform the categorical columns
categorical_cols = df_DROP_INVALID_AfterFS_ohe.select_dtypes(include=['object']).columns
df_DROP_INVALID_AfterFS_encoded = pd.DataFrame(ohe.fit_transform(df_DROP_INVALID_AfterFS_ohe[categorical_cols]))

# Get feature names for the encoded columns
encoded_feature_names = list(ohe.get_feature_names_out(categorical_cols))
df_DROP_INVALID_AfterFS_encoded.columns = encoded_feature_names

# Drop original categorical columns from the dataframe
df_DROP_INVALID_AfterFS_ohe = df_DROP_INVALID_AfterFS_ohe.drop(categorical_cols, axis=1)

# Concatenate the encoded columns with the remaining numerical features
df_DROP_INVALID_AfterFS_ohe = pd.concat([df_DROP_INVALID_AfterFS_ohe, df_DROP_INVALID_AfterFS_encoded], axis=1)

df_DROP_INVALID_AfterFS_ohe.head()

,UMUR (BULAN),banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price,STATUS PEMAKANAN_Malpemakanan,STATUS PEMAKANAN_Normal,"JENIS TASKA_TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, YPKT)",JENIS TASKA_TASKA Di Rumah,JENIS TASKA_TASKA Di Tempat Kerja (Sektor Awam),JENIS TASKA_TASKA Di Tempat Kerja (Sektor Swasta),JENIS TASKA_TASKA Institusi,TASKA_LOKASI_BANDAR,TASKA_LOKASI_LUAR BANDAR
0,60,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,53,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,52,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,58,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4,56,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [16]:
with open("ohe_encoder.pkl", "wb") as f:
    pickle.dump(ohe, f)

In [81]:
x_ohe = df_DROP_INVALID_AfterFS_ohe
y_ohe = df['BMI']

x_train_ohe, x_test_ohe, y_train_ohe, y_test_ohe = train_test_split(x_ohe, y_ohe, test_size = 0.30, random_state=42)

In [18]:
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(bootstrap=False, class_weight=None, max_depth=10, max_features='sqrt', min_samples_leaf=4,
                                  min_samples_split=10, n_estimators=100)
rf_model.fit(x_train_ohe, y_train_ohe)

# Make predictions on the test set
predictions = rf_model.predict(x_test_ohe)

In [19]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_ohe, predictions))

print("\nClassification Report:")
print(classification_report(y_test_ohe, predictions))

Confusion Matrix:
[[71  4  9]
 [13  9  5]
 [22  1 33]]

Classification Report:
                    precision    recall  f1-score   support

  Berat berlebihan       0.67      0.85      0.75        84
Kurang berat badan       0.64      0.33      0.44        27
            Normal       0.70      0.59      0.64        56

          accuracy                           0.68       167
         macro avg       0.67      0.59      0.61       167
      weighted avg       0.68      0.68      0.66       167



In [9]:
# Save the trained model to a file
with open('rf_model_3labels.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

In [11]:
df.to_csv('Dataset/rf_model_3labels.csv', index=False)

## 2 Labels (Binary)

In [68]:
df = df_DROP_INVALID_AfterFS.replace({'BMI': {"Berat badan normal": "Normal",
                                                 "Berlebihan berat badan": "Abnormal",
                                                 "Obes": "Abnormal",
                                                 "Risiko berlebihan berat badan": "Abnormal",
                                                 "Susut": "Abnormal",
                                                 "Susut teruk": "Abnormal"}})

In [69]:
df_DROP_INVALID_AfterFS_ohe = df.copy()
df_DROP_INVALID_AfterFS_ohe = df_DROP_INVALID_AfterFS_ohe.drop('BMI', axis=1)

In [70]:
# Create a OneHotEncoder object
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform the categorical columns
categorical_cols = df_DROP_INVALID_AfterFS_ohe.select_dtypes(include=['object']).columns
df_DROP_INVALID_AfterFS_encoded = pd.DataFrame(ohe.fit_transform(df_DROP_INVALID_AfterFS_ohe[categorical_cols]))

# Get feature names for the encoded columns
encoded_feature_names = list(ohe.get_feature_names_out(categorical_cols))
df_DROP_INVALID_AfterFS_encoded.columns = encoded_feature_names

# Drop original categorical columns from the dataframe
df_DROP_INVALID_AfterFS_ohe = df_DROP_INVALID_AfterFS_ohe.drop(categorical_cols, axis=1)

# Concatenate the encoded columns with the remaining numerical features
df_DROP_INVALID_AfterFS_ohe = pd.concat([df_DROP_INVALID_AfterFS_ohe, df_DROP_INVALID_AfterFS_encoded], axis=1)

df_DROP_INVALID_AfterFS_ohe.head()

,UMUR (BULAN),banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price,STATUS PEMAKANAN_Malpemakanan,STATUS PEMAKANAN_Normal,"JENIS TASKA_TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, YPKT)",JENIS TASKA_TASKA Di Rumah,JENIS TASKA_TASKA Di Tempat Kerja (Sektor Awam),JENIS TASKA_TASKA Di Tempat Kerja (Sektor Swasta),JENIS TASKA_TASKA Institusi,TASKA_LOKASI_BANDAR,TASKA_LOKASI_LUAR BANDAR
0,60,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,53,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,52,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,58,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4,56,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [71]:
x_ohe = df_DROP_INVALID_AfterFS_ohe
y_ohe = df['BMI']

x_train_ohe, x_test_ohe, y_train_ohe, y_test_ohe = train_test_split(x_ohe, y_ohe, test_size = 0.30, random_state=42)

In [72]:
# Initialize and train the Random Forest Classifier
rf_model = RandomForestClassifier(bootstrap=False, class_weight=None, max_depth=10, max_features='sqrt', min_samples_leaf=4,
                                  min_samples_split=10, n_estimators=100)
rf_model.fit(x_train_ohe, y_train_ohe)

# Make predictions on the test set
predictions = rf_model.predict(x_test_ohe)

In [73]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test_ohe, predictions))

print("\nClassification Report:")
print(classification_report(y_test_ohe, predictions))

Confusion Matrix:
[[100  11]
 [ 28  28]]

Classification Report:
              precision    recall  f1-score   support

    Abnormal       0.78      0.90      0.84       111
      Normal       0.72      0.50      0.59        56

    accuracy                           0.77       167
   macro avg       0.75      0.70      0.71       167
weighted avg       0.76      0.77      0.75       167



# Stratified K-Fold on 3 Labels RF Model

In [96]:
file_path = "D:/hp/Documents/Practicum/Code/Dataset/rf_model_3labels.csv"

# Read the first line to count the number of columns
with open(file_path, 'r', encoding='utf-8') as x:
    ncols = len(x.readline().strip().split(','))

# Load CSV using the correct number of columns
df_rf_model_3labels = pd.read_csv(file_path, usecols=range(0, ncols))

# Display the DataFrame
df_rf_model_3labels.head()

,UMUR (BULAN),STATUS PEMAKANAN,JENIS TASKA,TASKA_LOKASI,BMI,banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price
0,60,Malpemakanan,TASKA Institusi,BANDAR,Berat berlebihan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
1,53,Malpemakanan,TASKA Di Rumah,BANDAR,Berat berlebihan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
2,52,Malpemakanan,TASKA Institusi,BANDAR,Berat berlebihan,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9
3,58,Malpemakanan,"TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, Y...",LUAR BANDAR,Berat berlebihan,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9
4,56,Malpemakanan,"TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, Y...",LUAR BANDAR,Normal,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9


In [97]:
df = df_rf_model_3labels.copy()
df = df.drop('BMI', axis=1)

In [99]:
# Create a OneHotEncoder object
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform the categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns
df_encoded = pd.DataFrame(ohe.fit_transform(df[categorical_cols]))

# Get feature names for the encoded columns
encoded_feature_names = list(ohe.get_feature_names_out(categorical_cols))
df_encoded.columns = encoded_feature_names

# Drop original categorical columns from the dataframe
df = df.drop(categorical_cols, axis=1)

# Concatenate the encoded columns with the remaining numerical features
df = pd.concat([df, df_encoded], axis=1)

df.head()

,UMUR (BULAN),banana_price,papaya_price,rice_price,bread_price,fish_price,chicken_price,carrot_price,tomato_price,cauliflower_price,milk_price,STATUS PEMAKANAN_Malpemakanan,STATUS PEMAKANAN_Normal,"JENIS TASKA_TASKA Agensi Kerajaan (GENIUS, KEMAS, JPNIN, YPKT)",JENIS TASKA_TASKA Di Rumah,JENIS TASKA_TASKA Di Tempat Kerja (Sektor Awam),JENIS TASKA_TASKA Di Tempat Kerja (Sektor Swasta),JENIS TASKA_TASKA Institusi,TASKA_LOKASI_BANDAR,TASKA_LOKASI_LUAR BANDAR
0,60,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,53,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,52,6.0,4.29,25.99,2.8,9.99,8.9,4.49,5.00,7.9,20.9,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,58,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4,56,6.0,4.50,25.90,2.8,10.00,8.9,5.00,5.49,8.0,20.9,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [100]:
x_ohe = df
y_ohe = df_rf_model_3labels['BMI']

x_train_ohe, x_test_ohe, y_train_ohe, y_test_ohe = train_test_split(x_ohe, y_ohe, test_size = 0.30, random_state=42)

In [101]:
# Setup model
rf_model = RandomForestClassifier(
    bootstrap=False, class_weight=None, max_depth=10, max_features='sqrt',
    min_samples_leaf=4, min_samples_split=10, n_estimators=100
)

In [102]:
# Setup Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [106]:
# Store fold-level metrics
report_sum = defaultdict(lambda: defaultdict(float))
accuracy_sum = 0.0
n_folds = skf.get_n_splits()

In [107]:
fold = 1

for train_idx, test_idx in skf.split(x_train_ohe, y_train_ohe):
    X_train_fold = x_train_ohe.iloc[train_idx]
    X_test_fold = x_train_ohe.iloc[test_idx]
    y_train_fold = y_train_ohe.iloc[train_idx]
    y_test_fold = y_train_ohe.iloc[test_idx]
    
    rf_model.fit(X_train_fold, y_train_fold)
    preds = rf_model.predict(X_test_fold)
    
    acc = accuracy_score(y_test_fold, preds)
    accuracies.append(acc)

    print(f"\nFold {fold} Accuracy: {acc:.4f}")
    report = classification_report(y_test_fold, preds, output_dict=True)
    print(f"Classification Report (Fold {fold}):")
    print(pd.DataFrame(report).T)
    
    # Sum reports
    for label, metrics in report.items():
        if label == 'accuracy':
            accuracy_sum += metrics
        else:
            for metric_name, value in metrics.items():
                report_sum[label][metric_name] += value
    fold += 1


Fold 1 Accuracy: 0.7308
Classification Report (Fold 1):
                    precision    recall  f1-score    support
Berat berlebihan     0.702128  0.846154  0.767442  39.000000
Kurang berat badan   0.666667  0.461538  0.545455  13.000000
Normal               0.818182  0.692308  0.750000  26.000000
accuracy             0.730769  0.730769  0.730769   0.730769
macro avg            0.728992  0.666667  0.687632  78.000000
weighted avg         0.734902  0.730769  0.724630  78.000000

Fold 2 Accuracy: 0.6923
Classification Report (Fold 2):
                    precision    recall  f1-score    support
Berat berlebihan     0.702128  0.846154  0.767442  39.000000
Kurang berat badan   0.800000  0.285714  0.421053  14.000000
Normal               0.653846  0.680000  0.666667  25.000000
accuracy             0.692308  0.692308  0.692308   0.692308
macro avg            0.718658  0.603956  0.618387  78.000000
weighted avg         0.704220  0.692308  0.672970  78.000000

Fold 3 Accuracy: 0.7143
Classif

In [109]:
# Compute average classification report
print("\nAverage Classification Report Over All Folds:")
avg_report = {}
for label, metrics in report_sum.items():
    avg_report[label] = {}
    for metric_name, total_value in metrics.items():
        avg_report[label][metric_name] = total_value / n_folds

# Add average accuracy
avg_report['accuracy'] = {'score': accuracy_sum / n_folds}

print(pd.DataFrame(avg_report).T)



Average Classification Report Over All Folds:
                    precision    recall  f1-score  support    score
Berat berlebihan     0.684913  0.805128  0.739983     39.0      NaN
Kurang berat badan   0.718333  0.426374  0.527877     13.2      NaN
Normal               0.700382  0.650462  0.673504     25.2      NaN
macro avg            0.701209  0.627321  0.647121     77.4      NaN
weighted avg         0.696043  0.689810  0.682000     77.4      NaN
accuracy                  NaN       NaN       NaN      NaN  0.68981
